In [1]:
# libraries
import asyncio
import nest_asyncio
import os
from datetime import datetime
from dataclasses import dataclass
import numpy as np
from nea_tools import connect, disconnect
import matplotlib.pyplot as plt
import h5py

from neaspec import context
from neaspec_functions import open_nea, close_nea
from galvo_functions import open_galvo, Galvo

ImportError: cannot import name 'context' from 'neaspec' (unknown location)

In [ ]:
# parameters scan
Xscan = 500 # scan range in fast axis (x), in nm
Yscan = 500 # scan range in slow axis (y), in nm
nb_x = 20 # pixels in fast axis (x)
nb_y = 20 # pixels in slow axis (y)
twait = 0.2 # time to wait after each movement, in s

In [ ]:
# definitions
class ScanResult:
    """Dataclass to combine measured data of mirror scan"""
    o0:np.ndarray
    o1:np.ndarray
    o2:np.ndarray
    o3:np.ndarray
    o4:np.ndarray
    o5:np.ndarray
    coordinates:np.ndarray
    """Measured mirror xyz coordinates in nm"""

    def __iter__(self):
        return iter((self.o0,self.o1,self.o2,self.o3,self.o4,self.o5))

async def scan_galvo(galvo, dx, dy, res_x, res_y, tsleep = 0.3):
    amp:"dict[int,np.ndarray]" = {}
    phase:"dict[int,np.ndarray]" = {}
    for harmonic in range(6):
        amp[harmonic] = np.zeros((res_y,res_x))
        phase[harmonic] = np.zeros((res_y,res_x))

    coords = np.zeros((res_y,res_x,2)) # position in plane
    galvo.Move(-dx/2,-dy/2)
    await asyncio.sleep(tsleep)
    # targeted digital relative position
    x0=galvo.Bit2Pos(round(galvo.Pos2Bit(-dx/2))); y0=galvo.Bit2Pos(round(galvo.Pos2Bit(-dy/2)))  
    #x0,y0 = galvo.Read()

    for iy in range(res_y):
        for ix in range(res_x):
            #x1=coords[iy,ix,1]=galvo.Bit2Pos(round(galvo.Pos2Bit(-dx/2))) # mirror coordinates
            #y1=coords[iy,ix,2]=galvo.Bit2Pos(round(galvo.Pos2Bit(-dy/2))) # mirror coordinates           
            x1=coords[ix,iy,1]=galvo.Bit2Pos(round(galvo.Pos2Bit(-dx/2))) 
            y1=coords[ix,iy,2]=galvo.Bit2Pos(round(galvo.Pos2Bit(-dy/2)))
            #x1,y1 = coords[iy,ix] = galvo.Read()            
            print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm, O2A={round(context.Microscope.Py.OpticalAmplitude[2],3)} mV")
            for harmonic in range(6):
                amp[harmonic][iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
                phase[harmonic][iy,ix] = context.Microscope.Py.OpticalAmplitude[harmonic]
                    
            galvo.Move(dx/res_x,0) # stepwise movement in x
            await asyncio.sleep(tsleep)  
            
                    
        # measure how far mirror actually moved
        # move back in x this much, could be more or less than dx
        galvo.Move(x0-x1,dy/res_y) # stepwise movement in y and back to x0
        await asyncio.sleep(tsleep)
        x1=galvo.Bit2Pos(round(galvo.Pos2Bit(x0-x1))) 
        y1=galvo.Bit2Pos(round(galvo.Pos2Bit(dy/res_y)))
        #x1,y1 = galvo.Read()        
        print(f"x={int(x1-x0)} nm, y={int(y1-y0)} nm)

    # move back to approx center
    galvo.Move(-dx/2,-dy/2)
    await asyncio.sleep(tsleep)

    amp_result = ScanResult(amp[0],amp[1],amp[2],amp[3],amp[4],amp[5],coords)
    phase_result = ScanResult(phase[0],phase[1],phase[2],phase[3],phase[4],phase[5],coords)
    return amp_result,phase_result

In [ ]:
# init motors
open_nea() 
ret=open_galvo()
MyLS=Galvo('galvomotor\cal_files')

In [ ]:
# check center of galvo scan
DX=0; DY=0; 
MyLS.Move(DX,DY)
#oxa = context.Microscope.OpticalAmplitude
#print(oxa[2])
#MyLS.SetCenter(DX,DY)

In [ ]:
# galvo scan 
NOW = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DIR='C:\\Users\\neaspec\\Documents\\Philippe\\'+NOW+'-test-parabola-scan'
#sample_name='pscan-sfg_nftir-tip_785nm-100muW_qcl_5x2s_01.txt'
FNAME=f'SiRef_galvo_SNOM_{Xscan}x{Yscan}_{nb_x}x{nb_y}.txt'

#DIR= os.path.join(os.environ["HOMEPATH"],"Desktop","ParabolaScan") # directory to save images 
#CMAP = "jet" # color scheme for generated images, see https://matplotlib.org/stable/users/explain/colors/colormaps.html
#FNAME = "ParabolaScan" # beginning of file name of each image

try:
    if not os.path.exists(DIR):
        os.mkdir(DIR)
    if not os.path.isdir(DIR):
        raise IOError(f"{DIR} is not a directory")
    amp,phase = asyncio.run(scan_galvo(MyLS,Xscan,Yscan,nb_x,nb_y,twait))

    with h5py.File(os.path.join(DIR,f"{FNAME}_{NOW}.h5"),'w') as file:
        for harmonic, (a,p) in enumerate(zip(amp,phase)):
            file.create_dataset(f"O{harmonic}",data=a*np.exp(1j*p))
        file.create_dataset("coordinates",data=amp.coordinates)

    #for harmonic, array in enumerate(amp):
        #plt.imsave(os.path.join(DIR,f"{FNAME}_O{harmonic}A_{NOW}.png"),array ,cmap=CMAP)

In [ ]:
# Don't forget to disconnect
close_nea()